# Notebook 5. Cấp 7 (PUCT + mạng policy, value) và RQ3

Người phụ trách: Nguyễn Trọng Phúc.

Nội dung:
1. Huấn luyện mạng value trên ván chuyên gia của nb3.
2. Chọn hệ số `mix` (trộn giá trị mạng value và giá trị mô phỏng) bằng ván thử riêng.
3. Ablation cấp 7: MCTS thuần, thêm prior, thêm value, đầy đủ (cùng một hàm, khác hai công tắc).
4. RQ3: cấp 7 so với cấp 4 ở cùng số mô phỏng và cùng thời gian; cấp 7 so với cấp 5.

Thành phần AlphaGo đã hiện thực: mạng policy cho tiên nghiệm, mạng policy nhanh cho mô phỏng, mạng value, tìm kiếm cây PUCT, trộn value và mô phỏng theo `mix`.
Chưa hiện thực: mạng rollout tuyến tính riêng, tìm kiếm song song, mạng value học từ ván tự chơi của mạng RL quy mô lớn.

Cần có: `files/games_*.p` (nb3) và `files/policy_conn.keras` (nb3).
**Lưu ý:** mục 4b chạy theo thời gian thật, lúc đó không chạy việc nặng khác trên máy.

In [ ]:
# QUICK = True: chạy thử vài ván, vài epoch để kiểm tra code (ghi vào thư mục con thu_nhanh/)
# QUICK = False: chạy thật theo config.yaml, kết quả dùng cho báo cáo
QUICK = False

import os
if not os.path.exists("config.yaml"):
    raise RuntimeError("Hãy mở Jupyter trong thư mục final_connect_four (nơi có config.yaml).")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.experiments import setup, summaries_table, with_random_opening, move_times_of
from utils.arena import run_match, format_summary, read_log, first_player_stats
from utils.data import machine_info
import keras
from utils import data, nets, tactics
from utils.experiments import plot_scores, info_mean, records_of
ex = setup(QUICK)
cfg = ex.cfg
print("Cấu hình máy:", machine_info())

## 1. Mạng value

Mẫu = (mọi thế cờ của ván chuyên gia, kết quả cuối ván nhìn từ người sắp đi: hòa / thắng / thua). Chia train/val/test theo ván như nb3.
Kiến trúc chương 15 của sách, đầu ra 3 lớp; giá trị thế cờ = P(thắng) − P(thua).

In [ ]:
paths = []
for part in cfg["data"]["seeds"]:
    path = ex.model(f"games_{part}.p")
    if os.path.exists(path):
        paths.append(path)
    else:
        print("CẢNH BÁO: thiếu", path)
games = data.merge_game_files(paths)
print("Số ván:", len(games))

value_set = data.make_dataset(games, kind="value", seed=0)
print(data.describe_dataset(value_set))
counts = np.bincount(value_set["y_train"], minlength=3)
print("Nhãn train [hòa, người sắp đi thắng, người sắp đi thua]:", counts.tolist())
majority = int(np.argmax(counts))
value_baseline = float(np.mean(value_set["y_test"] == majority))
print(f"Mốc 'luôn đoán nhãn nhiều nhất': độ chính xác trên test = {value_baseline:.3f}")

In [ ]:
value_cfg = cfg["train_value"]
keras.utils.set_random_seed(0)
value_model, value_report = nets.train_value(games, path=ex.model("value_conn.keras"),
                                             epochs=ex.n(value_cfg["epochs"], 2),
                                             batch_size=value_cfg["batch_size"],
                                             learning_rate=value_cfg["learning_rate"], seed=0, verbose=2)
print(f"Huấn luyện {value_report['train_seconds']} giây | val acc {value_report['val_accuracy']:.3f} | "
      f"test acc {value_report['test_accuracy']:.3f} (mốc: {value_baseline:.3f})")
print("Số tham số của mạng value:", value_model.count_params())

In [ ]:
curve = value_report["curve"]
epochs = range(1, len(curve["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, curve["loss"], label="train")
axes[0].plot(epochs, curve["val_loss"], label="val")
axes[0].set_title("Loss")
axes[1].plot(epochs, curve["accuracy"], label="train")
axes[1].plot(epochs, curve["val_accuracy"], label="val")
axes[1].axhline(value_baseline, color="#999999", linestyle="--", label="luôn đoán một nhãn")
axes[1].set_title("Độ chính xác")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle("Đường cong học của mạng value")
fig.tight_layout()
fig.savefig(ex.fig("nb5_duong_cong_value.png"), dpi=150)

## 2. Chọn hệ số mix bằng ván thử riêng

`mix = 0`: chỉ dùng mạng value ở lá (nhanh). `mix = 0.5`: nửa mạng value, nửa mô phỏng nhanh 10 nước bằng mạng policy (như AlphaGo, chậm hơn nhiều).
Hai cấu hình đấu nhau với seed riêng (`mix_pilot.seed`), không dùng ván của RQ3 hay giải đấu cuối để chọn.

Ô dưới tự chọn `mix` theo quy tắc ghi sẵn và các mục sau của notebook này dùng giá trị đó.
**Trước khi chạy nb6:** ghi giá trị đã chọn vào `agents.cap7.params.mix` trong `config.yaml` để giải đấu dùng cùng cấu hình.

In [ ]:
pilot_cfg = cfg["mix_pilot"]
mixes = pilot_cfg["mixes"]
n_games = ex.n(pilot_cfg["games"], 4)
log_path = ex.log("nb5_chon_mix.jsonl")
name_a = f"cap7_mix{mixes[0]}"
name_b = f"cap7_mix{mixes[1]}"
a = ex.agent("cap7", overrides={"mix": mixes[0]})
b = ex.agent("cap7", overrides={"mix": mixes[1]})
pilot = run_match(a, b, name_a, name_b, n_games, seed=pilot_cfg["seed"], log_path=log_path)
print(format_summary(pilot))
print(f"Thời gian TB mỗi nước: {name_a} {pilot['time_mean'] * 1000:.0f} ms, "
      f"{name_b} {pilot['opponent_time_mean'] * 1000:.0f} ms")
# quy tắc: chỉ chọn mix chậm hơn khi nó mạnh hơn rõ rệt (khoảng tin cậy nằm hẳn dưới 0,5)
if pilot["ci_high"] < 0.5:
    chosen_mix = mixes[1]
    print(f"Kết luận: chọn mix = {mixes[1]} (mạnh hơn rõ rệt).")
else:
    chosen_mix = mixes[0]
    print(f"Kết luận: chọn mix = {mixes[0]} (không thua rõ rệt và nhanh hơn).")
print("Giá trị mix trong config.yaml:", cfg["agents"]["cap7"]["params"]["mix"])
if cfg["agents"]["cap7"]["params"]["mix"] != chosen_mix:
    print(f"NHỚ: sửa agents.cap7.params.mix thành {chosen_mix} trong config.yaml trước khi chạy nb6.")

## 3. Ablation cấp 7

Bốn cấu hình cùng 200 mô phỏng, cùng đấu một đối thủ chung (`rq3.opponent`, cấp 3). Mỗi ván mở đầu bằng 2 nước ngẫu nhiên có seed.

In [ ]:
rq3_cfg = cfg["rq3"]
n_games = ex.n(rq3_cfg["games_per_pair"], 4)
opening = rq3_cfg["opening_random_moves"]
log_path = ex.log("nb5_ablation_cap7.jsonl")
opponent_name = rq3_cfg["opponent"]
opponent = with_random_opening(ex.agent(opponent_name), opening)

ablation_summaries = []
for k, name in enumerate(["mcts_thuan", "them_prior", "them_value", "day_du"]):
    agent = with_random_opening(ex.agent(name, section="rq3_ablation"), opening)
    s = run_match(agent, opponent, name, opponent_name, n_games, seed=rq3_cfg["seed"] + k, log_path=log_path)
    ablation_summaries.append(s)
    print(format_summary(s))

ablation_table = summaries_table(ablation_summaries)
ablation_table.to_csv(ex.table("nb5_ablation_cap7.csv"), index=False, encoding="utf-8-sig")
ablation_table

## 4. RQ3

### 4a. Cùng số mô phỏng mỗi nước

In [ ]:
sim_list = rq3_cfg["simulations"] if not QUICK else [20, 50]
log_path = ex.log("nb5_rq3_mo_phong.jsonl")
sim_summaries = []
for k, sims in enumerate(sim_list):
    name7 = f"cap7_{sims}mp"
    name4 = f"cap4_{sims}mp"
    a = ex.agent("cap7", overrides={"num_simulations": sims, "mix": chosen_mix})
    b = ex.agent("cap4", overrides={"num_simulations": sims})
    start = time.perf_counter()
    s = run_match(a, b, name7, name4, n_games, seed=rq3_cfg["seed"] + 10 + k, log_path=log_path)
    sim_summaries.append(s)
    print(format_summary(s), "| chạy mất", round(time.perf_counter() - start), "giây")

sim_table = summaries_table(sim_summaries)
sim_table.insert(1, "mô phỏng/nước", sim_list)
sim_table.to_csv(ex.table("nb5_rq3_mo_phong.csv"), index=False, encoding="utf-8-sig")
sim_table

### 4b. Cùng thời gian mỗi nước

Cấp 7 gọi mạng ở mỗi lá nên mỗi mô phỏng tốn hơn cấp 4 nhiều lần; cùng thời gian thì cấp 7 làm được ít mô phỏng hơn. Bảng ghi số mô phỏng trung bình của mỗi bên.

In [ ]:
time_limits = rq3_cfg["time_limits"] if not QUICK else [0.05, 0.1]
log_path = ex.log("nb5_rq3_thoi_gian.jsonl")
time_summaries = []
extra = []
for k, t in enumerate(time_limits):
    name7 = f"cap7_{t}s"
    name4 = f"cap4_{t}s"
    a = ex.agent("cap7", overrides={"time_limit": t, "mix": chosen_mix})
    b = ex.agent("cap4", overrides={"time_limit": t})
    start = time.perf_counter()
    s = run_match(a, b, name7, name4, n_games, seed=rq3_cfg["seed"] + 20 + k, log_path=log_path)
    time_summaries.append(s)
    records = records_of(log_path, name7, name4)
    extra.append({"giây/nước": t,
                  "mô phỏng TB cấp 7": info_mean(records, name7, "simulations"),
                  "mô phỏng TB cấp 4": info_mean(records, name4, "simulations")})
    print(format_summary(s), "| chạy mất", round(time.perf_counter() - start), "giây")

time_table = pd.concat([summaries_table(time_summaries), pd.DataFrame(extra)], axis=1)
time_table.to_csv(ex.table("nb5_rq3_thoi_gian.csv"), index=False, encoding="utf-8-sig")
time_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, x_values, series, xlabel in [(axes[0], sim_list, sim_summaries, "số mô phỏng mỗi nước"),
                                     (axes[1], time_limits, time_summaries, "thời gian mỗi nước (giây)")]:
    scores = [s["score"] for s in series]
    low = [s["score"] - s["ci_low"] for s in series]
    high = [s["ci_high"] - s["score"] for s in series]
    ax.errorbar(x_values, scores, yerr=[low, high], marker="o", capsize=4, linewidth=2)
    ax.axhline(0.5, color="#999999", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("score của cấp 7 khi đấu cấp 4")
    ax.grid(alpha=0.25)
axes[0].set_title("Cùng số mô phỏng")
axes[1].set_title("Cùng thời gian")
fig.suptitle("RQ3: cấp 7 đấu cấp 4")
fig.tight_layout()
fig.savefig(ex.fig("nb5_rq3.png"), dpi=150)

### 4c. Cấp 7 so với cấp 5 và độ chính xác chiến thuật

Cấu hình mặc định của mỗi cấp (theo `config.yaml`).

In [ ]:
log_path = ex.log("nb5_rq3_cap5.jsonl")
cap7 = with_random_opening(ex.agent("cap7", overrides={"mix": chosen_mix}), opening)
cap5 = with_random_opening(ex.agent("cap5"), opening)
s = run_match(cap7, cap5, "cap7", "cap5", n_games, seed=rq3_cfg["seed"] + 30, log_path=log_path)
print(format_summary(s))
summaries_table([s]).to_csv(ex.table("nb5_rq3_cap7_cap5.csv"), index=False, encoding="utf-8-sig")

In [ ]:
tactics_path = ex.model("tactics.json")
if os.path.exists(tactics_path):
    positions = tactics.load_tactics(tactics_path)
    rows = []
    for name in ["cap4", "cap5", "cap7"]:
        if name == "cap7":
            move_fn, info_fn = ex.agent(name, overrides={"mix": chosen_mix})
        else:
            move_fn, info_fn = ex.agent(name)
        row = {"tác tử": name}
        row.update(tactics.score_agent(move_fn, positions))
        rows.append(row)
    for name in ["mcts_thuan", "them_prior", "them_value", "day_du"]:
        move_fn, info_fn = ex.agent(name, section="rq3_ablation")
        row = {"tác tử": name}
        row.update(tactics.score_agent(move_fn, positions))
        rows.append(row)
    rq3_tactics = pd.DataFrame(rows).round(3)
    rq3_tactics.to_csv(ex.table("nb5_chien_thuat_cap7.csv"), index=False, encoding="utf-8-sig")
    display(rq3_tactics)
else:
    print("Chưa có", tactics_path, "- chạy mục 1 của nb2 (hoặc chép file từ máy kia) rồi chạy lại ô này.")